In [ ]:
cd ../..

## Feature Ideas

- For the following time periods:
    - All time
    - Prev 3 months
    - 3 - 6 months in the past
    - 6 - 9 months in the past
- Calculate the following features:
- `price`: only for all time
- `category`: only for all time
- `title`: Embedding, only for all time
- `weeks_since_first_review`: only for all time
- `weeks_since_median_review`: only for all time
- `weeks_since_latest_review`: only for all time
- `last_review_summary`: Embedding, only for all time
- `last_review_is_verified`: only for all time
- `num_reviews`
- `sum_ratings`
- `avg_rating`
- `std_rating`
- `min_rating`
- `max_rating`
- `avg_review_length`
- `pct_verified_reviews`
- `product_bias`: avg_rating - global_avg_rating / global_std_rating
- reviewer aggs:
    - `avg_reviewer_total_spent`
    - `avg_reviewer_num_reviews`
    - `avg_reviewer_avg_price`
    - `avg_reviewer_avg_rating`
    - `avg_reviewer_std_rating`

In [ ]:
import duckdb
from torch_frame.utils import infer_df_stype

import utils

conn = duckdb.connect('amazon/amazon.db')
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

In [ ]:
with open('amazon/item-ltv/feats.sql', 'r') as f:
    # run once with train_labels and once with val_labels
    template = f.read()

# create train, val and test features
# takes 1 - 5 mins
for s in ['train', 'val', 'test']:
    print(f'Creating {s} table')
    query = utils.render_jinja_sql(template, dict(set=s, subsample=0))
    conn.sql(query)
    print(f'{s} table created')

In [ ]:
utils.validate_feature_tables('item_ltv', conn)

In [ ]:
%%sql train_df <<
from product_ltv_train_feats

In [ ]:
infer_df_stype(train_df)

In [ ]:
utils.feature_summary_df(train_df.sample(20_000), 'ltv', classification=False)

## Feature Importances

In [ ]:
import numpy as np
import shap
from torch_frame import TaskType, stype
from torch_frame.data import Dataset
from torch_frame.gbdt import LightGBM

from inferred_stypes import task_to_stypes
from train_gbdt import TASK_PARAMS

TASK = 'rel-amazon-item-ltv'

task_params = TASK_PARAMS[TASK]

In [ ]:
%%sql val_df <<
select * from item_ltv_val_feats;

In [ ]:
col_to_stype = task_to_stypes[TASK].copy()
del col_to_stype['title']
del col_to_stype['last_review_summary']
val_tf = Dataset(
    val_df,
    col_to_stype=col_to_stype,
    target_col=task_params['target_col'],
).materialize().tensor_frame

In [ ]:
gbdt = LightGBM(task_type=task_params['task_type'])
gbdt.load(f'models/{TASK}_lgbm.json')
pred = gbdt.predict(tf_test=val_tf).numpy()

In [ ]:
shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

sample = np.random.randint(0, len(val_tf), size=10_000)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[sample])
shap_values = explainer.shap_values(val_arr, pred[sample])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=30, feature_names=feat_names)

In [ ]:
shap.dependence_plot('price', shap_values, val_arr, feature_names=feat_names, xmax=60, ymin=-100, ymax=100)

In [ ]:
shap.dependence_plot('avg_reviewer_num_reviews', shap_values, val_arr, feature_names=feat_names)

In [ ]:
shap.dependence_plot('num_reviews_0_to_3', shap_values, val_arr, feature_names=feat_names, xmax=100, ymax=200)

In [ ]:
conn.close()